
# EdgeMLP Pipeline

This notebook assumes the dataset is already cleaned and feature engineered.
It does **not** redo EDA or the earlier preprocessing pipeline.

It runs four experiments:
1. **EdgeMLP without feature-engineered columns**
2. **EdgeMLP with feature-engineered columns**
3. **EdgeMLP with feature-engineered columns + XGBoost stacker**
4. **EdgeMLP with feature-engineered columns + CatBoost stacker**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

!pip -q install xgboost catboost

import copy
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


In [4]:
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("pandas:", pd.__version__)

numpy: 2.0.2
torch: 2.10.0+cu128
pandas: 2.2.2


In [5]:
base_path = "/content/drive/MyDrive/dataset_cleaned/HI-Medium_Trans.csv"  # change if needed

df = pd.read_csv(base_path)
print("Shape:", df.shape)
df.head()

Shape: (31898218, 21)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0,8.824035,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0,8.954194,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0,7.539681,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0,18.119128,2022-09-01 00:03:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0,17.641288,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0


## Preparation for model

In [6]:

REQUIRED_COLS = [
    "Timestamp", "From Bank", "Account", "To Bank", "Account.1",
    "Amount Received", "Receiving Currency", "Amount Paid",
    "Payment Currency", "Payment Format", "Is Laundering",
    "Log Amount Received", "tx_hour", "tx_dow", "tx_month", "tx_day",
    "tx_date", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"
]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

model_df = df[REQUIRED_COLS].copy()

model_df["_ts"] = pd.to_datetime(model_df["Timestamp"], errors="coerce")
model_df = model_df.dropna(subset=["_ts"]).copy()

model_df["Is Laundering"] = pd.to_numeric(model_df["Is Laundering"], errors="coerce")
model_df = model_df[model_df["Is Laundering"].isin([0, 1])].copy()
model_df["Is Laundering"] = model_df["Is Laundering"].astype(np.float32)

numeric_to_coerce = [
    "Amount Received", "Amount Paid", "Log Amount Received",
    "tx_hour", "tx_dow", "tx_month", "tx_day", "tx_is_weekend",
    "tx_hour_sin", "tx_hour_cos"
]
for col in numeric_to_coerce:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

if pd.api.types.is_numeric_dtype(model_df["tx_date"]):
    model_df["tx_date_num"] = pd.to_numeric(model_df["tx_date"], errors="coerce")
else:
    tx_date_parsed = pd.to_datetime(model_df["tx_date"], errors="coerce")
    if tx_date_parsed.notna().any():
        origin = tx_date_parsed.min()
        model_df["tx_date_num"] = (tx_date_parsed - origin).dt.days.astype(float)
    else:
        model_df["tx_date_num"] = np.nan

for col in ["From Bank", "Account", "To Bank", "Account.1", "Receiving Currency", "Payment Currency", "Payment Format"]:
    model_df[col] = model_df[col].astype(str).fillna("UNK")

model_df = model_df.sort_values("_ts").reset_index(drop=True)
model_df["row_id"] = np.arange(len(model_df), dtype=np.int64)

print("Rows after modeling setup:", len(model_df))
print("Positive rate:", float(model_df["Is Laundering"].mean()))
print(model_df[["_ts", "Is Laundering"]].head())


Rows after modeling setup: 31898218
Positive rate: 0.0011044504353776574
         _ts  Is Laundering
0 2022-09-01            0.0
1 2022-09-01            0.0
2 2022-09-01            0.0
3 2022-09-01            0.0
4 2022-09-01            0.0


In [7]:

N = len(model_df)
n_train = int(0.70 * N)
n_val = int(0.15 * N)

TRAIN_IDX = np.arange(0, n_train, dtype=np.int64)
VAL_IDX = np.arange(n_train, n_train + n_val, dtype=np.int64)
TEST_IDX = np.arange(n_train + n_val, N, dtype=np.int64)

print("Split sizes:", len(TRAIN_IDX), len(VAL_IDX), len(TEST_IDX))
print("Train pos rate:", float(model_df.loc[TRAIN_IDX, "Is Laundering"].mean()))
print("Val pos rate:", float(model_df.loc[VAL_IDX, "Is Laundering"].mean()))
print("Test pos rate:", float(model_df.loc[TEST_IDX, "Is Laundering"].mean()))


Split sizes: 22328752 4784732 4784734
Train pos rate: 0.0009442085865885019
Val pos rate: 0.0011467726435512304
Test pos rate: 0.0018099229782819748


In [8]:

MAX_ROWS = None

if MAX_ROWS is not None and len(model_df) > MAX_ROWS:
    work_df = model_df.iloc[:MAX_ROWS].copy()
else:
    work_df = model_df.copy()

N_WORK = len(work_df)
n_train_w = int(0.70 * N_WORK)
n_val_w = int(0.15 * N_WORK)
TR_EDGE = np.arange(0, n_train_w, dtype=np.int64)
VA_EDGE = np.arange(n_train_w, n_train_w + n_val_w, dtype=np.int64)
TE_EDGE = np.arange(n_train_w + n_val_w, N_WORK, dtype=np.int64)

print("Rows used in experiments:", N_WORK)
print("Edge split sizes:", len(TR_EDGE), len(VA_EDGE), len(TE_EDGE))

Rows used in experiments: 31898218
Edge split sizes: 22328752 4784732 4784734


In [9]:

def build_node_arrays(df_in):
    src_df = df_in[["From Bank", "Account"]].copy()
    dst_df = df_in[["To Bank", "Account.1"]].rename(columns={"To Bank": "From Bank", "Account.1": "Account"}).copy()
    all_nodes = pd.concat([src_df, dst_df], ignore_index=True)

    codes, uniques = pd.factorize(list(map(tuple, all_nodes.to_numpy())), sort=False)
    m = len(df_in)
    src = codes[:m].astype(np.int64)
    dst = codes[m:].astype(np.int64)
    return src, dst, len(uniques)

SRC, DST, NUM_NODES = build_node_arrays(work_df)
Y = work_df["Is Laundering"].to_numpy(dtype=np.float32)

print("Num nodes:", NUM_NODES)
print("Num edges:", len(SRC))

Num nodes: 2077023
Num edges: 31898218


In [10]:

RAW_NUMERIC_COLS = [
    "Amount Received",
    "Amount Paid",
    "Log Amount Received",
]

ENGINEERED_NUMERIC_COLS = [
    "tx_hour",
    "tx_dow",
    "tx_month",
    "tx_day",
    "tx_date_num",
    "tx_is_weekend",
    "tx_hour_sin",
    "tx_hour_cos",
]

TABULAR_CAT_COLS = [
    "From Bank",
    "To Bank",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
]

In [11]:
def fit_category_maps(train_df, cat_cols):
    maps = {}
    cardinalities = {}
    for col in cat_cols:
        values = train_df[col].astype(str).fillna("UNK")
        uniq = pd.Index(values.unique())
        mapping = {val: idx + 1 for idx, val in enumerate(uniq)}
        maps[col] = mapping
        cardinalities[col] = len(mapping) + 1
    return maps, cardinalities


def apply_category_maps(df_in, cat_cols, maps):
    out = []
    for col in cat_cols:
        mapped = df_in[col].astype(str).map(maps[col]).fillna(0).astype(np.int64).to_numpy()
        out.append(mapped)
    return np.stack(out, axis=1) if out else np.zeros((len(df_in), 0), dtype=np.int64)


def build_numeric_matrix(df_in, numeric_cols, means=None, stds=None):
    X = df_in[numeric_cols].apply(pd.to_numeric, errors="coerce").copy()
    if means is None:
        means = X.mean()
    X = X.fillna(means)
    if stds is None:
        stds = X.std(ddof=0).replace(0, 1.0)
    X = (X - means) / stds
    return X.to_numpy(dtype=np.float32), means, stds


## EdgeMLP

In [12]:
class EdgeMLPWithFeatures(nn.Module):
    def __init__(self, num_nodes, num_numeric, cat_cardinalities, node_dim=64, cat_dim=16, hidden_dim=256, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, node_dim)
        self.dropout = nn.Dropout(dropout)

        self.cat_embeddings = nn.ModuleList()
        cat_out_dim = 0
        for card in cat_cardinalities:
            emb_dim = min(cat_dim, max(4, int(math.sqrt(card)) + 1))
            self.cat_embeddings.append(nn.Embedding(card, emb_dim))
            cat_out_dim += emb_dim

        input_dim = 4 * node_dim + num_numeric + cat_out_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, src, dst, x_num, x_cat):
        hs = self.dropout(self.node_emb(src))
        hd = self.dropout(self.node_emb(dst))
        pair_vec = torch.cat([hs, hd, torch.abs(hs - hd), hs * hd], dim=1)

        pieces = [pair_vec]
        if x_num.shape[1] > 0:
            pieces.append(x_num)
        if len(self.cat_embeddings) > 0:
            cat_vecs = []
            for i, emb in enumerate(self.cat_embeddings):
                cat_vecs.append(emb(x_cat[:, i]))
            pieces.append(torch.cat(cat_vecs, dim=1))

        x = torch.cat(pieces, dim=1)
        return self.mlp(x).squeeze(1)


In [13]:
def tensorize_inputs(df_in, numeric_cols, cat_cols, cat_maps=None, means=None, stds=None):
    if cat_maps is None:
        cat_maps, cat_cards = fit_category_maps(df_in.iloc[TR_EDGE], cat_cols)
    else:
        cat_cards = {c: max(cat_maps[c].values(), default=0) + 1 for c in cat_cols}

    X_cat = apply_category_maps(df_in, cat_cols, cat_maps)
    X_num, means, stds = build_numeric_matrix(df_in, numeric_cols, means=means, stds=stds)
    cat_cardinalities = [cat_cards[c] for c in cat_cols]
    return X_num, X_cat, means, stds, cat_maps, cat_cardinalities


def train_single_edgemlp_run(
    src_np,
    dst_np,
    y_np,
    x_num_np,
    x_cat_np,
    train_idx,
    val_idx,
    cat_cardinalities,
    num_nodes,
    epochs=6,
    batch_size=131072,
    lr=1e-3,
    weight_decay=1e-5,
):
    model = EdgeMLPWithFeatures(
        num_nodes=num_nodes,
        num_numeric=x_num_np.shape[1],
        cat_cardinalities=cat_cardinalities,
        node_dim=64,
        cat_dim=16,
        hidden_dim=256,
        dropout=0.2,
    ).to(DEVICE)

    src_t = torch.from_numpy(src_np).long().to(DEVICE)
    dst_t = torch.from_numpy(dst_np).long().to(DEVICE)
    y_t = torch.from_numpy(y_np).float().to(DEVICE)
    x_num_t = torch.from_numpy(x_num_np).float().to(DEVICE)
    x_cat_t = torch.from_numpy(x_cat_np).long().to(DEVICE)

    pos_rate = float(y_np[train_idx].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_pr = -np.inf
    history = []

    for epoch in range(epochs):
        model.train()
        order = train_idx.copy()
        np.random.shuffle(order)
        running_loss = 0.0

        for start in range(0, len(order), batch_size):
            b_idx = order[start:start + batch_size]
            b = torch.from_numpy(b_idx).long().to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(src_t[b], dst_t[b], x_num_t[b], x_cat_t[b])
            loss = loss_fn(logits, y_t[b])
            loss.backward()
            optimizer.step()

            running_loss += float(loss.detach().cpu()) * len(b_idx)

        val_pred = predict_edgemlp(model, src_t, dst_t, x_num_t, x_cat_t, val_idx, batch_size=batch_size)
        val_pr = average_precision_score(y_np[val_idx], val_pred)
        val_roc = roc_auc_score(y_np[val_idx], val_pred)
        epoch_loss = running_loss / len(order)
        history.append({"epoch": epoch + 1, "train_loss": epoch_loss, "val_pr_auc": val_pr, "val_roc_auc": val_roc})
        print(f"Epoch {epoch + 1:02d} | loss={epoch_loss:.6f} | val_pr={val_pr:.6f} | val_roc={val_roc:.6f}")

        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


@torch.no_grad()
def predict_edgemlp(model, src_t, dst_t, x_num_t, x_cat_t, idx_np, batch_size=131072):
    model.eval()
    preds = np.empty(len(idx_np), dtype=np.float32)
    for start in range(0, len(idx_np), batch_size):
        b_idx = idx_np[start:start + batch_size]
        b = torch.from_numpy(b_idx).long().to(DEVICE)
        logits = model(src_t[b], dst_t[b], x_num_t[b], x_cat_t[b])
        preds[start:start + len(b_idx)] = torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
    return preds


In [14]:
def run_edgemlp_experiment(name, numeric_cols, cat_cols, epochs=6, batch_size=131072):
    X_num, X_cat, means, stds, cat_maps, cat_cardinalities = tensorize_inputs(
        work_df,
        numeric_cols=numeric_cols,
        cat_cols=cat_cols,
    )

    model, history = train_single_edgemlp_run(
        src_np=SRC,
        dst_np=DST,
        y_np=Y,
        x_num_np=X_num,
        x_cat_np=X_cat,
        train_idx=TR_EDGE,
        val_idx=VA_EDGE,
        cat_cardinalities=cat_cardinalities,
        num_nodes=NUM_NODES,
        epochs=epochs,
        batch_size=batch_size,
    )

    src_t = torch.from_numpy(SRC).long().to(DEVICE)
    dst_t = torch.from_numpy(DST).long().to(DEVICE)
    x_num_t = torch.from_numpy(X_num).float().to(DEVICE)
    x_cat_t = torch.from_numpy(X_cat).long().to(DEVICE)

    val_pred = predict_edgemlp(model, src_t, dst_t, x_num_t, x_cat_t, VA_EDGE, batch_size=batch_size)
    test_pred = predict_edgemlp(model, src_t, dst_t, x_num_t, x_cat_t, TE_EDGE, batch_size=batch_size)

    metrics = {
        "model": name,
        "val_roc_auc": roc_auc_score(Y[VA_EDGE], val_pred),
        "val_pr_auc": average_precision_score(Y[VA_EDGE], val_pred),
        "test_roc_auc": roc_auc_score(Y[TE_EDGE], test_pred),
        "test_pr_auc": average_precision_score(Y[TE_EDGE], test_pred),
    }

    print(name)
    for k, v in metrics.items():
        if k != "model":
            print(f"{k}: {v:.6f}")

    artifact = {
        "name": name,
        "model": model,
        "history": history,
        "metrics": metrics,
        "numeric_cols": numeric_cols,
        "cat_cols": cat_cols,
        "cat_maps": cat_maps,
        "cat_cardinalities": cat_cardinalities,
        "means": means,
        "stds": stds,
        "X_num": X_num,
        "X_cat": X_cat,
        "val_pred": val_pred,
        "test_pred": test_pred,
    }
    return artifact

In [15]:
raw_feature_run = run_edgemlp_experiment(
    name="EdgeMLP_raw_features_only",
    numeric_cols=RAW_NUMERIC_COLS,
    cat_cols=TABULAR_CAT_COLS,
    epochs=6,
)

Epoch 01 | loss=0.682080 | val_pr=0.014711 | val_roc=0.949904
Epoch 02 | loss=0.499696 | val_pr=0.016866 | val_roc=0.954482
Epoch 03 | loss=0.463290 | val_pr=0.022316 | val_roc=0.959819
Epoch 04 | loss=0.411023 | val_pr=0.080903 | val_roc=0.962024
Epoch 05 | loss=0.353394 | val_pr=0.143941 | val_roc=0.960661
Epoch 06 | loss=0.290036 | val_pr=0.161564 | val_roc=0.954851
EdgeMLP_raw_features_only
val_roc_auc: 0.954851
val_pr_auc: 0.161564
test_roc_auc: 0.951951
test_pr_auc: 0.126576


In [16]:
feature_engineered_run = run_edgemlp_experiment(
    name="EdgeMLP_raw_plus_engineered_features",
    numeric_cols=RAW_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS,
    cat_cols=TABULAR_CAT_COLS,
    epochs=6,
)


Epoch 01 | loss=0.751201 | val_pr=0.016996 | val_roc=0.952197
Epoch 02 | loss=0.479525 | val_pr=0.019439 | val_roc=0.956064
Epoch 03 | loss=0.435056 | val_pr=0.027523 | val_roc=0.960163
Epoch 04 | loss=0.370985 | val_pr=0.079372 | val_roc=0.961162
Epoch 05 | loss=0.307025 | val_pr=0.148844 | val_roc=0.958342
Epoch 06 | loss=0.249385 | val_pr=0.172532 | val_roc=0.952482
EdgeMLP_raw_plus_engineered_features
val_roc_auc: 0.952482
val_pr_auc: 0.172532
test_roc_auc: 0.955403
test_pr_auc: 0.163310


## EdgeMLP + Feature Engineering

In [17]:
def build_oof_edge_scores(
    numeric_cols,
    cat_cols,
    n_folds=5,
    epochs=5,
    batch_size=131072,
):
    X_num, X_cat, means, stds, cat_maps, cat_cardinalities = tensorize_inputs(
        work_df,
        numeric_cols=numeric_cols,
        cat_cols=cat_cols,
    )

    src_t = torch.from_numpy(SRC).long().to(DEVICE)
    dst_t = torch.from_numpy(DST).long().to(DEVICE)
    x_num_t = torch.from_numpy(X_num).float().to(DEVICE)
    x_cat_t = torch.from_numpy(X_cat).long().to(DEVICE)

    edge_score = np.full(len(work_df), np.nan, dtype=np.float32)
    train_blocks = np.array_split(TR_EDGE, n_folds)

    for k in range(n_folds):
        holdout = train_blocks[k]
        if k == 0:
            print(f"Fold {k + 1}/{n_folds}: skipped earliest block because there is no earlier training data.")
            continue

        train_fold = np.concatenate(train_blocks[:k])
        model_k, _ = train_single_edgemlp_run(
            src_np=SRC,
            dst_np=DST,
            y_np=Y,
            x_num_np=X_num,
            x_cat_np=X_cat,
            train_idx=train_fold,
            val_idx=holdout,
            cat_cardinalities=cat_cardinalities,
            num_nodes=NUM_NODES,
            epochs=epochs,
            batch_size=batch_size,
        )
        edge_score[holdout] = predict_edgemlp(model_k, src_t, dst_t, x_num_t, x_cat_t, holdout, batch_size=batch_size)
        print(f"Fold {k + 1}/{n_folds}: done")

    final_model, _ = train_single_edgemlp_run(
        src_np=SRC,
        dst_np=DST,
        y_np=Y,
        x_num_np=X_num,
        x_cat_np=X_cat,
        train_idx=TR_EDGE,
        val_idx=VA_EDGE,
        cat_cardinalities=cat_cardinalities,
        num_nodes=NUM_NODES,
        epochs=epochs,
        batch_size=batch_size,
    )

    earliest_missing = TR_EDGE[np.isnan(edge_score[TR_EDGE])]
    if len(earliest_missing) > 0:
        edge_score[earliest_missing] = predict_edgemlp(final_model, src_t, dst_t, x_num_t, x_cat_t, earliest_missing, batch_size=batch_size)

    edge_score[VA_EDGE] = predict_edgemlp(final_model, src_t, dst_t, x_num_t, x_cat_t, VA_EDGE, batch_size=batch_size)
    edge_score[TE_EDGE] = predict_edgemlp(final_model, src_t, dst_t, x_num_t, x_cat_t, TE_EDGE, batch_size=batch_size)

    print("Missing edge scores:", int(np.isnan(edge_score).sum()))

    return {
        "edge_score": edge_score,
        "final_model": final_model,
        "X_num": X_num,
        "X_cat": X_cat,
        "means": means,
        "stds": stds,
        "cat_maps": cat_maps,
        "cat_cardinalities": cat_cardinalities,
    }


In [18]:
stacking_source = build_oof_edge_scores(
    numeric_cols=RAW_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS,
    cat_cols=TABULAR_CAT_COLS,
    n_folds=5,
    epochs=5,
)

stack_df = work_df.copy()
stack_df["edge_score"] = stacking_source["edge_score"]
stack_df = stack_df.dropna(subset=["edge_score"]).reset_index(drop=True)

stack_df.head()

Fold 1/5: skipped earliest block because there is no earlier training data.
Epoch 01 | loss=0.515153 | val_pr=0.002191 | val_roc=0.770492
Epoch 02 | loss=0.372975 | val_pr=0.003981 | val_roc=0.881613
Epoch 03 | loss=0.275900 | val_pr=0.006179 | val_roc=0.907887
Epoch 04 | loss=0.243072 | val_pr=0.007084 | val_roc=0.913926
Epoch 05 | loss=0.231529 | val_pr=0.007355 | val_roc=0.913713
Fold 2/5: done
Epoch 01 | loss=0.692194 | val_pr=0.010067 | val_roc=0.923992
Epoch 02 | loss=0.438826 | val_pr=0.012503 | val_roc=0.941052
Epoch 03 | loss=0.396362 | val_pr=0.013123 | val_roc=0.944058
Epoch 04 | loss=0.370638 | val_pr=0.014414 | val_roc=0.946069
Epoch 05 | loss=0.345718 | val_pr=0.015668 | val_roc=0.946808
Fold 3/5: done
Epoch 01 | loss=0.834993 | val_pr=0.009719 | val_roc=0.921430
Epoch 02 | loss=0.505257 | val_pr=0.015380 | val_roc=0.943156
Epoch 03 | loss=0.431379 | val_pr=0.017471 | val_roc=0.949392
Epoch 04 | loss=0.391532 | val_pr=0.022911 | val_roc=0.951464
Epoch 05 | loss=0.343796 |

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Log Amount Received,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos,_ts,tx_date_num,row_id,edge_score
0,2022/09/01 00:00,3184493,846486F60,1187482,8464841F0,4224.24,Shekel,4224.24,Shekel,ACH,0.0,8.348831,0,3,9,1,2022-09-01,0,0.0,1.0,2022-09-01,0.0,0,2.637555e-01
1,2022/09/01 00:00,269579,82091AAB0,269579,82091AAB0,599177.46,US Dollar,599177.46,US Dollar,Reinvestment,0.0,13.303315,0,3,9,1,2022-09-01,0,0.0,1.0,2022-09-01,0.0,1,4.605238e-09
2,2022/09/01 00:00,28348,803C292C0,28348,803C292C0,24.53,US Dollar,24.53,US Dollar,Reinvestment,0.0,3.239854,0,3,9,1,2022-09-01,0,0.0,1.0,2022-09-01,0.0,2,3.787189e-13
3,2022/09/01 00:00,218354,806D51900,28313,806D52970,1850.26,US Dollar,1850.26,US Dollar,ACH,0.0,7.523622,0,3,9,1,2022-09-01,0,0.0,1.0,2022-09-01,0.0,3,3.579720e-01
4,2022/09/01 00:00,1108042,8430139A0,1108042,8430139A0,55129.17,UK Pound,55129.17,UK Pound,Reinvestment,0.0,10.917452,0,3,9,1,2022-09-01,0,0.0,1.0,2022-09-01,0.0,4,5.013837e-07


In [19]:
STACK_NUMERIC_COLS = ["edge_score"] + RAW_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS
STACK_CAT_COLS = TABULAR_CAT_COLS
STACK_FEATURE_COLS = STACK_CAT_COLS + STACK_NUMERIC_COLS

n_stack = len(stack_df)
n_train_s = int(0.70 * n_stack)
n_val_s = int(0.15 * n_stack)

stack_train = stack_df.iloc[:n_train_s].copy()
stack_val = stack_df.iloc[n_train_s:n_train_s + n_val_s].copy()
stack_test = stack_df.iloc[n_train_s + n_val_s:].copy()

print(stack_train.shape, stack_val.shape, stack_test.shape)

(22328752, 24) (4784732, 24) (4784734, 24)


## EdgeMLP + XGBoost

In [20]:
def one_hot_align(train_df, val_df, test_df, cat_cols, num_cols):
    high_card_exclude = {"Account", "Account.1", "Timestamp", "_ts", "tx_date"}

    safe_cat_cols = [
        c for c in cat_cols
        if c in train_df.columns
        and c not in high_card_exclude
        and train_df[c].nunique(dropna=False) <= 1000
    ]

    safe_num_cols = [c for c in num_cols if c in train_df.columns]

    x_train = pd.get_dummies(
        train_df[safe_cat_cols + safe_num_cols],
        columns=safe_cat_cols,
        dummy_na=False,
        dtype=np.uint8,
    )
    x_val = pd.get_dummies(
        val_df[safe_cat_cols + safe_num_cols],
        columns=safe_cat_cols,
        dummy_na=False,
        dtype=np.uint8,
    )
    x_test = pd.get_dummies(
        test_df[safe_cat_cols + safe_num_cols],
        columns=safe_cat_cols,
        dummy_na=False,
        dtype=np.uint8,
    )

    x_val = x_val.reindex(columns=x_train.columns, fill_value=0)
    x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

    for c in safe_num_cols:
        if c in x_train.columns:
            x_train[c] = pd.to_numeric(x_train[c], errors="coerce").fillna(0).astype(np.float32)
            x_val[c] = pd.to_numeric(x_val[c], errors="coerce").fillna(0).astype(np.float32)
            x_test[c] = pd.to_numeric(x_test[c], errors="coerce").fillna(0).astype(np.float32)

    print(f"XGBoost safe categorical cols used: {safe_cat_cols}")
    print(f"XGBoost safe numeric cols used: {safe_num_cols}")
    print("Xtr shape:", x_train.shape, "Xva shape:", x_val.shape, "Xte shape:", x_test.shape)

    return x_train, x_val, x_test


Xtr_xgb, Xva_xgb, Xte_xgb = one_hot_align(
    stack_train, stack_val, stack_test, STACK_CAT_COLS, STACK_NUMERIC_COLS
)

ytr_xgb = stack_train["Is Laundering"].astype(int).to_numpy()
yva_xgb = stack_val["Is Laundering"].astype(int).to_numpy()
yte_xgb = stack_test["Is Laundering"].astype(int).to_numpy()

scale_pos_weight = (len(ytr_xgb) - ytr_xgb.sum()) / max(ytr_xgb.sum(), 1)

xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    tree_method="hist",
    n_jobs=-1,
)

xgb_model.fit(
    Xtr_xgb,
    ytr_xgb,
    eval_set=[(Xva_xgb, yva_xgb)],
    verbose=100,
)

pva_xgb = xgb_model.predict_proba(Xva_xgb)[:, 1]
pte_xgb = xgb_model.predict_proba(Xte_xgb)[:, 1]

xgb_metrics = {
    "model": "EdgeMLP_plus_XGBoost",
    "val_roc_auc": roc_auc_score(yva_xgb, pva_xgb),
    "val_pr_auc": average_precision_score(yva_xgb, pva_xgb),
    "test_roc_auc": roc_auc_score(yte_xgb, pte_xgb),
    "test_pr_auc": average_precision_score(yte_xgb, pte_xgb),
}

xgb_metrics

XGBoost safe categorical cols used: ['Receiving Currency', 'Payment Currency', 'Payment Format']
XGBoost safe numeric cols used: ['edge_score', 'Amount Received', 'Amount Paid', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_month', 'tx_day', 'tx_date_num', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']
Xtr shape: (22328752, 49) Xva shape: (4784732, 49) Xte shape: (4784734, 49)
[0]	validation_0-aucpr:0.01212
[100]	validation_0-aucpr:0.11460
[200]	validation_0-aucpr:0.12289
[300]	validation_0-aucpr:0.12874
[400]	validation_0-aucpr:0.13350
[500]	validation_0-aucpr:0.13382
[600]	validation_0-aucpr:0.13394
[699]	validation_0-aucpr:0.13406


{'model': 'EdgeMLP_plus_XGBoost',
 'val_roc_auc': np.float64(0.9694303834818467),
 'val_pr_auc': np.float64(0.13429392368931867),
 'test_roc_auc': np.float64(0.9770299907471521),
 'test_pr_auc': np.float64(0.24477021775776323)}

## EdgeMLP + CatBoost

In [21]:
# Keep original variable names so nothing else breaks
Xtr_cb = stack_train[STACK_FEATURE_COLS].copy()
Xva_cb = stack_val[STACK_FEATURE_COLS].copy()
Xte_cb = stack_test[STACK_FEATURE_COLS].copy()

ytr_cb = stack_train["Is Laundering"].astype(int).to_numpy()
yva_cb = stack_val["Is Laundering"].astype(int).to_numpy()
yte_cb = stack_test["Is Laundering"].astype(int).to_numpy()

# Exclude problematic high-cardinality / ID-like columns for CatBoost too
high_card_exclude = {"Account", "Account.1", "Timestamp", "_ts", "tx_date"}

safe_cb_cat_cols = [
    c for c in STACK_CAT_COLS
    if c in Xtr_cb.columns and c not in high_card_exclude
]

safe_cb_num_cols = [
    c for c in Xtr_cb.columns
    if c not in safe_cb_cat_cols and c not in high_card_exclude
]

Xtr_cb = Xtr_cb[safe_cb_cat_cols + safe_cb_num_cols].copy()
Xva_cb = Xva_cb[safe_cb_cat_cols + safe_cb_num_cols].copy()
Xte_cb = Xte_cb[safe_cb_cat_cols + safe_cb_num_cols].copy()

for c in safe_cb_cat_cols:
    Xtr_cb[c] = Xtr_cb[c].astype(str).fillna("MISSING")
    Xva_cb[c] = Xva_cb[c].astype(str).fillna("MISSING")
    Xte_cb[c] = Xte_cb[c].astype(str).fillna("MISSING")

for c in safe_cb_num_cols:
    Xtr_cb[c] = pd.to_numeric(Xtr_cb[c], errors="coerce").fillna(0).astype(np.float32)
    Xva_cb[c] = pd.to_numeric(Xva_cb[c], errors="coerce").fillna(0).astype(np.float32)
    Xte_cb[c] = pd.to_numeric(Xte_cb[c], errors="coerce").fillna(0).astype(np.float32)

cat_idx = [Xtr_cb.columns.get_loc(c) for c in safe_cb_cat_cols]

print(f"CatBoost safe categorical cols used: {safe_cb_cat_cols}")
print(f"CatBoost safe numeric cols used: {safe_cb_num_cols}")
print("Xtr_cb shape:", Xtr_cb.shape, "Xva_cb shape:", Xva_cb.shape, "Xte_cb shape:", Xte_cb.shape)

cat_model = CatBoostClassifier(
    iterations=2500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=200,
    task_type="CPU",
)

cat_model.fit(
    Xtr_cb,
    ytr_cb,
    eval_set=(Xva_cb, yva_cb),
    cat_features=cat_idx,
    use_best_model=True,
)

pva_cb = cat_model.predict_proba(Xva_cb)[:, 1]
pte_cb = cat_model.predict_proba(Xte_cb)[:, 1]

cat_metrics = {
    "model": "EdgeMLP_plus_CatBoost",
    "val_roc_auc": roc_auc_score(yva_cb, pva_cb),
    "val_pr_auc": average_precision_score(yva_cb, pva_cb),
    "test_roc_auc": roc_auc_score(yte_cb, pte_cb),
    "test_pr_auc": average_precision_score(yte_cb, pte_cb),
}

cat_metrics

CatBoost safe categorical cols used: ['From Bank', 'To Bank', 'Receiving Currency', 'Payment Currency', 'Payment Format']
CatBoost safe numeric cols used: ['edge_score', 'Amount Received', 'Amount Paid', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_month', 'tx_day', 'tx_date_num', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']
Xtr_cb shape: (22328752, 17) Xva_cb shape: (4784732, 17) Xte_cb shape: (4784734, 17)
0:	learn: 0.9316023	test: 0.9468027	best: 0.9468027 (0)	total: 6.21s	remaining: 4h 18m 46s
200:	learn: 0.9806080	test: 0.9812169	best: 0.9812254 (199)	total: 41m 7s	remaining: 7h 50m 21s
400:	learn: 0.9838401	test: 0.9819914	best: 0.9820243 (390)	total: 1h 22m 59s	remaining: 7h 14m 26s
600:	learn: 0.9867302	test: 0.9821105	best: 0.9822553 (537)	total: 2h 9m 44s	remaining: 6h 49m 57s
800:	learn: 0.9884848	test: 0.9814108	best: 0.9822553 (537)	total: 2h 55m 26s	remaining: 6h 12m 8s
1000:	learn: 0.9896310	test: 0.9811871	best: 0.9822553 (537)	total: 3h 41m 25s	remaining: 5h 31m 3

{'model': 'EdgeMLP_plus_CatBoost',
 'val_roc_auc': np.float64(0.9791702597141881),
 'val_pr_auc': np.float64(0.47409752945373873),
 'test_roc_auc': np.float64(0.9857349569515735),
 'test_pr_auc': np.float64(0.630389169097421)}

## Results comparison

In [22]:
results = pd.DataFrame([
    raw_feature_run["metrics"],
    feature_engineered_run["metrics"],
    xgb_metrics,
    cat_metrics,
])

results

,model,val_roc_auc,val_pr_auc,test_roc_auc,test_pr_auc
0,EdgeMLP_raw_features_only,0.954851,0.161564,0.951951,0.126576
1,EdgeMLP_raw_plus_engineered_features,0.952482,0.172532,0.955403,0.163310
2,EdgeMLP_plus_XGBoost,0.969430,0.134294,0.977030,0.244770
3,EdgeMLP_plus_CatBoost,0.979170,0.474098,0.985735,0.630389
